# The limma Algorithm: Step-by-Step Walkthrough

Based on:
- Ritchie et al. (2015), *Nucleic Acids Research*, 43(7), e47
- Smyth (2004), *Statistical Applications in Genetics and Molecular Biology*, 3, Article 3

The limma algorithm consists of the following main steps:
1. Set up the design matrix
2. Ordinary Least Squares (OLS) fit per gene (`lmFit`)
3. Empirical Bayes: estimate hyperparameters $d_0$ and $s_0^2$ (`eBayes`)
4. Compute moderated (posterior) variances
5. Compute moderated t-statistics
6. Compute p-values

## Imports and Data

In [6]:
import numpy as np
from scipy.special import digamma, polygamma
from scipy.optimize import brentq
from scipy.stats import t as t_dist, norm

In [7]:
gene_names = ["B3AT_39_51", "CRK_214_226", "EGFR_862_874"]

# Expression matrix: rows = genes, columns = samples (4 control, 4 treatment)
Y = np.array([
    [4.702532097102332, 3.252219123075958, 1.7478129764031984, 2.486398666237567,
     -6.643856189774724, -6.643856189774724, -0.3703684499451156, 0.35954238668714017],
    [5.43195088221519, 5.247927513443585, -3.142957953842043, -6.643856189774724,
     -6.643856189774724, -6.643856189774724, -6.643856189774724, -6.643856189774724],
    [4.872921659824102, 5.045218752156492, 3.5033007261479865, 1.857042046157957,
     1.3055025469742512, -6.643856189774724, 3.269259026374422, -6.643856189774724]
])

n_genes, n_samples = Y.shape
print(f"Number of genes:   {n_genes}")
print(f"Number of samples: {n_samples} (4 control + 4 treatment)")

Number of genes:   3
Number of samples: 8 (4 control + 4 treatment)


---
## Step 1: Design Matrix

The linear model for each gene $g$ is:

$$E(\mathbf{y}_g) = X \boldsymbol{\beta}_g$$

where:
- $\mathbf{y}_g$ is the vector of expression values for gene $g$ (length $n = 8$)
- $X$ is the $n \times p$ design matrix ($n = 8$, $p = 2$)
- $\boldsymbol{\beta}_g = (\beta_{g0}, \beta_{g1})^T$ is the coefficient vector

For our two-group design (Control vs Treatment):

$$
X = \begin{pmatrix}
1 & 0 \\ 1 & 0 \\ 1 & 0 \\ 1 & 0 \\
1 & 1 \\ 1 & 1 \\ 1 & 1 \\ 1 & 1
\end{pmatrix}
$$

- $\beta_{g0}$ = mean of the Control group
- $\beta_{g1}$ = difference Treatment − Control = **log-fold-change (logFC)**

Residual degrees of freedom: $d_g = n - p = 8 - 2 = 6$

In [8]:
# Design matrix: intercept + treatment indicator
X = np.array([
    [1, 0], [1, 0], [1, 0], [1, 0],  # Control
    [1, 1], [1, 1], [1, 1], [1, 1],  # Treatment
])

p = X.shape[1]  # number of parameters
d_g = n_samples - p  # residual degrees of freedom

print(f"Design matrix X:\n{X}\n")
print(f"Number of parameters p = {p}")
print(f"Residual degrees of freedom d_g = n - p = {n_samples} - {p} = {d_g}")

Design matrix X:
[[1 0]
 [1 0]
 [1 0]
 [1 0]
 [1 1]
 [1 1]
 [1 1]
 [1 1]]

Number of parameters p = 2
Residual degrees of freedom d_g = n - p = 8 - 2 = 6


---
## Step 2: Ordinary Least Squares per Gene (`lmFit`)

For each gene $g$, we fit a linear model via OLS:

**OLS estimator:**
$$\hat{\boldsymbol{\beta}}_g = (X^T X)^{-1} X^T \mathbf{y}_g$$

**Fitted values:**
$$\hat{\mathbf{y}}_g = X \hat{\boldsymbol{\beta}}_g$$

**Residuals:**
$$\mathbf{e}_g = \mathbf{y}_g - \hat{\mathbf{y}}_g$$

**Residual sum of squares:**
$$\text{RSS}_g = \mathbf{e}_g^T \mathbf{e}_g = \sum_j e_{gj}^2$$

**Gene-wise residual variance:**
$$s_g^2 = \frac{\text{RSS}_g}{d_g}$$

Under the standard linear model assumptions, $s_g^2$ is an unbiased estimator of $\sigma_g^2$, and $d_g \cdot s_g^2 / \sigma_g^2 \sim \chi^2_{d_g}$.

In [9]:
# Precompute (X^T X)^{-1} — needed for OLS and later for standard errors
XtX_inv = np.linalg.inv(X.T @ X)

print(f"(X^T X)^{{-1}} =\n{XtX_inv}\n")

(X^T X)^{-1} =
[[ 0.25 -0.25]
 [-0.25  0.5 ]]



In [10]:
# Fit OLS for each gene
beta_hat = np.zeros((n_genes, p))
sigma2_g = np.zeros(n_genes)

for g in range(n_genes):
    y_g = Y[g, :]

    # OLS coefficients
    beta_g = XtX_inv @ X.T @ y_g
    beta_hat[g, :] = beta_g

    # Residuals and variance
    residuals = y_g - X @ beta_g
    RSS = np.sum(residuals ** 2)
    sigma2_g[g] = RSS / d_g

    print(f"--- {gene_names[g]} ---")
    print(f"  beta_0 (Control mean)          = {beta_g[0]:.6f}")
    print(f"  beta_1 (logFC: Treat - Ctrl)   = {beta_g[1]:.6f}")
    print(f"  Residuals = {np.round(residuals, 4)}")
    print(f"  RSS_g     = {RSS:.6f}")
    print(f"  s^2_g     = RSS / {d_g} = {sigma2_g[g]:.6f}")
    print()

--- B3AT_39_51 ---
  beta_0 (Control mean)          = 3.047241
  beta_1 (logFC: Treat - Ctrl)   = -6.371875
  Residuals = [ 1.6553  0.205  -1.2994 -0.5608 -3.3192 -3.3192  2.9543  3.6842]
  RSS_g     = 49.120374
  s^2_g     = RSS / 6 = 8.186729

--- CRK_214_226 ---
  beta_0 (Control mean)          = 0.223266
  beta_1 (logFC: Treat - Ctrl)   = -6.867122
  Residuals = [ 5.2087  5.0247 -3.3662 -6.8671 -0.     -0.     -0.     -0.    ]
  RSS_g     = 110.866452
  s^2_g     = RSS / 6 = 18.477742

--- EGFR_862_874 ---
  beta_0 (Control mean)          = 3.819621
  beta_1 (logFC: Treat - Ctrl)   = -5.997858
  Residuals = [ 1.0533  1.2256 -0.3163 -1.9626  3.4837 -4.4656  5.4475 -4.4656]
  RSS_g     = 88.258470
  s^2_g     = RSS / 6 = 14.709745



In [11]:
# Summary of gene-wise variances
print("Gene-wise variances s^2_g:")
for g in range(n_genes):
    print(f"  {gene_names[g]}: {sigma2_g[g]:.6f}")

Gene-wise variances s^2_g:
  B3AT_39_51: 8.186729
  CRK_214_226: 18.477742
  EGFR_862_874: 14.709745


---
## Step 3: Empirical Bayes — Estimate Hyperparameters (`eBayes`)

This is the **core innovation of limma**. Instead of using the gene-wise variance $s_g^2$ directly (which is noisy with few samples), limma assumes all gene variances come from a common prior distribution and "borrows strength" across genes.

### The Hierarchical Model (Smyth 2004)

**Prior on the true gene variance:**
$$\frac{1}{\sigma_g^2} \sim \frac{1}{d_0 s_0^2} \chi^2_{d_0}$$

**Sampling distribution (likelihood):**
$$\frac{d_g \cdot s_g^2}{\sigma_g^2} \sim \chi^2_{d_g}$$

The two hyperparameters to estimate:
- $s_0^2$ = prior variance (the "global" variance all genes are shrunk towards)
- $d_0$ = prior degrees of freedom (controls shrinkage strength; large $d_0$ = strong shrinkage)

### Estimation via the marginal distribution of $\log(s_g^2)$

After integrating out $\sigma_g^2$, the marginal distribution of $\log(s_g^2)$ has:

$$E[\log(s_g^2)] = \log(s_0^2) + \psi(d_g/2) - \log(d_g/2) - \psi(d_0/2) + \log(d_0/2)$$

$$\text{Var}[\log(s_g^2)] = \psi'(d_g/2) + \psi'(d_0/2)$$

where $\psi$ is the digamma function and $\psi'$ is the trigamma function.

**Estimating $d_0$** by matching the variance equation:
$$\psi'(d_0/2) = \widehat{\text{Var}}[\log(s_g^2)] - \psi'(d_g/2)$$

Then invert the trigamma function to get $d_0/2$.

**Estimating $s_0^2$** by matching the mean equation:
$$\log(s_0^2) = \overline{\log(s_g^2)} - \psi(d_g/2) + \log(d_g/2) + \psi(d_0/2) - \log(d_0/2)$$

In [12]:
# Compute log(s^2_g) for each gene
log_s2 = np.log(sigma2_g)

print("log(s^2_g) values:")
for g in range(n_genes):
    print(f"  {gene_names[g]}: {log_s2[g]:.6f}")

mean_log_s2 = np.mean(log_s2)
var_log_s2 = np.var(log_s2, ddof=1)  # unbiased sample variance

print(f"\nMean of log(s^2_g)     = {mean_log_s2:.6f}")
print(f"Variance of log(s^2_g) = {var_log_s2:.6f}")

log(s^2_g) values:
  B3AT_39_51: 2.102514
  CRK_214_226: 2.916567
  EGFR_862_874: 2.688510

Mean of log(s^2_g)     = 2.569197
Variance of log(s^2_g) = 0.176347


### Step 3a: Estimate $d_0$

In [13]:
# Trigamma at d_g/2
trigamma_dg_half = polygamma(1, d_g / 2)  # psi'(3)
print(f"psi'(d_g/2) = psi'({d_g/2}) = {trigamma_dg_half:.6f}")

# Target value for psi'(d0/2)
target_trigamma = var_log_s2 - trigamma_dg_half
print(f"\npsi'(d0/2) = Var[log(s^2_g)] - psi'(d_g/2)")
print(f"           = {var_log_s2:.6f} - {trigamma_dg_half:.6f}")
print(f"           = {target_trigamma:.6f}")

psi'(d_g/2) = psi'(3.0) = 0.394934

psi'(d0/2) = Var[log(s^2_g)] - psi'(d_g/2)
           = 0.176347 - 0.394934
           = -0.218587


In [14]:
# Check if d0 is finite or infinite
if target_trigamma <= 0:
    d0 = np.inf
    print(f"psi'(d0/2) = {target_trigamma:.6f} <= 0")
    print(f"Since psi'(x) > 0 for all x > 0, no finite d0 exists.")
    print(f"=> d0 = infinity (full shrinkage to the prior)")
    print()
    print(f"NOTE: With only {n_genes} genes this is expected.")
    print(f"In practice with thousands of genes, d0 is typically finite.")
else:
    # Invert the trigamma function: find x such that psi'(x) = target_trigamma
    def trigamma_eq(x):
        return polygamma(1, x) - target_trigamma
    d0_half = brentq(trigamma_eq, 0.001, 1e6)
    d0 = 2 * d0_half
    print(f"psi'(d0/2) = {target_trigamma:.6f} > 0 => d0 is finite")
    print(f"d0/2 = {d0_half:.6f}")
    print(f"d0   = {d0:.6f}")

psi'(d0/2) = -0.218587 <= 0
Since psi'(x) > 0 for all x > 0, no finite d0 exists.
=> d0 = infinity (full shrinkage to the prior)

NOTE: With only 3 genes this is expected.
In practice with thousands of genes, d0 is typically finite.


### Step 3b: Estimate $s_0^2$

In [15]:
digamma_dg_half = digamma(d_g / 2)

if np.isinf(d0):
    # When d0 = inf: psi(d0/2) - log(d0/2) -> 0
    log_s0_sq = mean_log_s2 - digamma_dg_half + np.log(d_g / 2)
else:
    digamma_d0_half = digamma(d0 / 2)
    log_s0_sq = (mean_log_s2
                 - digamma_dg_half + np.log(d_g / 2)
                 + digamma_d0_half - np.log(d0 / 2))

s0_sq = np.exp(log_s0_sq)

print(f"psi(d_g/2) = psi({d_g/2}) = {digamma_dg_half:.6f}")
print(f"log(d_g/2) = log({d_g/2}) = {np.log(d_g / 2):.6f}")
print(f"\nlog(s0^2)  = {log_s0_sq:.6f}")
print(f"s0^2       = exp({log_s0_sq:.6f}) = {s0_sq:.6f}")

psi(d_g/2) = psi(3.0) = 0.922784
log(d_g/2) = log(3.0) = 1.098612

log(s0^2)  = 2.745025
s0^2       = exp(2.745025) = 15.565005


In [16]:
# Summary of Step 3
print("=" * 50)
print("Empirical Bayes hyperparameters:")
print(f"  d0   = {d0}")
print(f"  s0^2 = {s0_sq:.6f}")
print("=" * 50)

Empirical Bayes hyperparameters:
  d0   = inf
  s0^2 = 15.565005


---
## Step 4: Moderated (Posterior) Variances

The posterior distribution of $\sigma_g^2$ given $s_g^2$ is a scaled inverse chi-squared distribution. The posterior mean (= moderated variance) is:

$$\tilde{s}_g^2 = \frac{d_0 \, s_0^2 + d_g \, s_g^2}{d_0 + d_g}$$

This is a **weighted average** of:
- the prior variance $s_0^2$ (weight $d_0$)
- the observed gene-wise variance $s_g^2$ (weight $d_g$)

The associated posterior degrees of freedom are:

$$\tilde{d}_g = d_0 + d_g$$

When $d_0 = \infty$: $\tilde{s}_g^2 = s_0^2$ for all genes (complete shrinkage).

In [17]:
s2_post = np.zeros(n_genes)

for g in range(n_genes):
    if np.isinf(d0):
        s2_post[g] = s0_sq
    else:
        s2_post[g] = (d0 * s0_sq + d_g * sigma2_g[g]) / (d0 + d_g)

    direction = "down" if sigma2_g[g] > s2_post[g] else "up"
    print(f"{gene_names[g]}:")
    print(f"  s^2_g (OLS)        = {sigma2_g[g]:.6f}")
    print(f"  s^2*_g (moderated) = {s2_post[g]:.6f}  (shrunk {direction})")
    print()

# Posterior degrees of freedom
d_post = np.inf if np.isinf(d0) else d0 + d_g
print(f"Posterior degrees of freedom: d*_g = d0 + d_g = {d_post}")

B3AT_39_51:
  s^2_g (OLS)        = 8.186729
  s^2*_g (moderated) = 15.565005  (shrunk up)

CRK_214_226:
  s^2_g (OLS)        = 18.477742
  s^2*_g (moderated) = 15.565005  (shrunk down)

EGFR_862_874:
  s^2_g (OLS)        = 14.709745
  s^2*_g (moderated) = 15.565005  (shrunk up)

Posterior degrees of freedom: d*_g = d0 + d_g = inf


---
## Step 5: Moderated t-Statistics

**Ordinary t-statistic** (standard two-sample t-test):
$$t_g = \frac{\hat{\beta}_{g1}}{s_g \sqrt{c_{11}}}$$

**Moderated t-statistic** (limma replaces $s_g$ with $\tilde{s}_g$):
$$\tilde{t}_g = \frac{\hat{\beta}_{g1}}{\tilde{s}_g \sqrt{c_{11}}}$$

where $c_{11}$ is the $(1,1)$ diagonal element of $(X^T X)^{-1}$.

For our balanced two-group design with $n_1 = n_2 = 4$:
$$c_{11} = 0.5$$

This is equivalent to $\text{Var}(\hat{\beta}_1) = \sigma^2 (1/n_1 + 1/n_2) = \sigma^2 / 2$.

In [18]:
c_jj = XtX_inv[1, 1]  # variance factor for beta_1
print(f"c_11 = (X^T X)^{{-1}}[1,1] = {c_jj:.6f}")
print(f"sqrt(c_11) = {np.sqrt(c_jj):.6f}")
print()

t_ordinary = np.zeros(n_genes)
t_moderated = np.zeros(n_genes)

for g in range(n_genes):
    se_ordinary = np.sqrt(sigma2_g[g]) * np.sqrt(c_jj)
    se_moderated = np.sqrt(s2_post[g]) * np.sqrt(c_jj)

    t_ordinary[g] = beta_hat[g, 1] / se_ordinary
    t_moderated[g] = beta_hat[g, 1] / se_moderated

    print(f"--- {gene_names[g]} ---")
    print(f"  logFC = {beta_hat[g, 1]:.6f}")
    print(f"  Ordinary:  SE = {se_ordinary:.6f},  t  = {t_ordinary[g]:.6f}")
    print(f"  Moderated: SE = {se_moderated:.6f},  t~ = {t_moderated[g]:.6f}")
    print()

c_11 = (X^T X)^{-1}[1,1] = 0.500000
sqrt(c_11) = 0.707107

--- B3AT_39_51 ---
  logFC = -6.371875
  Ordinary:  SE = 2.023206,  t  = -3.149394
  Moderated: SE = 2.789714,  t~ = -2.284061

--- CRK_214_226 ---
  logFC = -6.867122
  Ordinary:  SE = 3.039551,  t  = -2.259255
  Moderated: SE = 2.789714,  t~ = -2.461587

--- EGFR_862_874 ---
  logFC = -5.997858
  Ordinary:  SE = 2.711987,  t  = -2.211610
  Moderated: SE = 2.789714,  t~ = -2.149991



---
## Step 6: P-Values

**Ordinary:**
$$t_g \sim t(d_g) \qquad p = 2 \cdot P(T > |t_g|), \quad T \sim t(d_g)$$

**Moderated:**
$$\tilde{t}_g \sim t(d_0 + d_g) \qquad p = 2 \cdot P(T > |\tilde{t}_g|), \quad T \sim t(d_0 + d_g)$$

Key advantage: the moderated t-statistics have **more degrees of freedom** ($d_0 + d_g$ instead of $d_g$), which gives a narrower t-distribution and more statistical power, especially with small sample sizes.

When $d_0 = \infty$: the t-distribution converges to the standard normal distribution.

In [19]:
print(f"Degrees of freedom:")
print(f"  Ordinary:  df = d_g = {d_g}")
print(f"  Moderated: df = d0 + d_g = {d_post}")
if np.isinf(d_post):
    print(f"  (df = inf => t-distribution becomes standard normal)")
print()

for g in range(n_genes):
    # Ordinary p-value
    p_ordinary = 2 * t_dist.sf(np.abs(t_ordinary[g]), df=d_g)

    # Moderated p-value
    if np.isinf(d_post):
        p_moderated = 2 * norm.sf(np.abs(t_moderated[g]))
    else:
        p_moderated = 2 * t_dist.sf(np.abs(t_moderated[g]), df=d_post)

    print(f"--- {gene_names[g]} ---")
    print(f"  Ordinary:  t = {t_ordinary[g]:+.4f},  df = {d_g},     p = {p_ordinary:.6e}")
    df_str = "inf" if np.isinf(d_post) else f"{d_post:.2f}"
    print(f"  Moderated: t~= {t_moderated[g]:+.4f},  df = {df_str},   p = {p_moderated:.6e}")
    print()

Degrees of freedom:
  Ordinary:  df = d_g = 6
  Moderated: df = d0 + d_g = inf
  (df = inf => t-distribution becomes standard normal)

--- B3AT_39_51 ---
  Ordinary:  t = -3.1494,  df = 6,     p = 1.982995e-02
  Moderated: t~= -2.2841,  df = inf,   p = 2.236797e-02

--- CRK_214_226 ---
  Ordinary:  t = -2.2593,  df = 6,     p = 6.461420e-02
  Moderated: t~= -2.4616,  df = inf,   p = 1.383239e-02

--- EGFR_862_874 ---
  Ordinary:  t = -2.2116,  df = 6,     p = 6.899013e-02
  Moderated: t~= -2.1500,  df = inf,   p = 3.155595e-02



---
## Summary Table (analogous to `topTable`)

In [20]:
print(f"{'Gene':<15} {'logFC':>10} {'s2_g':>10} {'s2*_g':>10} {'t_ord':>10} {'t_mod':>10} {'p_ord':>12} {'p_mod':>12}")
print("-" * 95)

for g in range(n_genes):
    p_ord = 2 * t_dist.sf(np.abs(t_ordinary[g]), df=d_g)
    p_mod = (2 * norm.sf(np.abs(t_moderated[g]))
             if np.isinf(d_post)
             else 2 * t_dist.sf(np.abs(t_moderated[g]), df=d_post))

    print(f"{gene_names[g]:<15} {beta_hat[g,1]:>10.4f} {sigma2_g[g]:>10.4f} {s2_post[g]:>10.4f} "
          f"{t_ordinary[g]:>10.4f} {t_moderated[g]:>10.4f} {p_ord:>12.6e} {p_mod:>12.6e}")

Gene                 logFC       s2_g      s2*_g      t_ord      t_mod        p_ord        p_mod
-----------------------------------------------------------------------------------------------
B3AT_39_51         -6.3719     8.1867    15.5650    -3.1494    -2.2841 1.982995e-02 2.236797e-02
CRK_214_226        -6.8671    18.4777    15.5650    -2.2593    -2.4616 6.461420e-02 1.383239e-02
EGFR_862_874       -5.9979    14.7097    15.5650    -2.2116    -2.1500 6.899013e-02 3.155595e-02


### Legend

| Column | Description |
|--------|-------------|
| logFC | log-fold-change (Treatment − Control) |
| s2_g | gene-wise variance (OLS) |
| s2*_g | moderated variance (Empirical Bayes posterior) |
| t_ord | ordinary t-statistic |
| t_mod | moderated t-statistic |
| p_ord | p-value from ordinary t-test (df = 6) |
| p_mod | p-value from moderated t-test (df = $d_0 + d_g$) |

## Alternative: InMoose

In [24]:
import pandas as pd
from inmoose.limma import lmFit, eBayes, topTable

In [25]:
# --- Daten vorbereiten ---
# Y: Matrix (Gene × Samples), log2-transformiert
# Zeilen = Gene/Peptide, Spalten = Samples
Y = np.array([
    [ 4.703,  3.252,  1.748,  2.486, -6.644, -6.644, -0.370,  0.360],  # B3AT
    [ 5.432,  5.248, -3.143, -6.644, -6.644, -6.644, -6.644, -6.644],  # CRK
    [ 4.873,  5.045,  3.503,  1.857,  1.306, -6.644,  3.269, -6.644],  # EGFR
])
gene_names = ["B3AT_39_51", "CRK_214_226", "EGFR_862_874"]

# --- Design-Matrix ---
# 4 Control, 4 Treatment
X = np.array([
    [1, 0],  # Control
    [1, 0],
    [1, 0],
    [1, 0],
    [1, 1],  # Treatment
    [1, 1],
    [1, 1],
    [1, 1],
])

# --- limma Pipeline ---
fit = lmFit(Y, X)
fit = eBayes(fit, robust=False)  # robust=True bei heterogenen Varianzen

# --- Ergebnisse extrahieren ---
# coef=1 = zweite Spalte der Design-Matrix (Treatment-Effekt / logFC)
results = topTable(fit, coef=1, number=len(gene_names), genelist=gene_names)
print(results)

KeyError: -2